# Notebook 1: Data Collection (Enhanced v2)
## Songwriter Style Analysis - High-Volume Data Collection

**Objective**: Collect 80-100 songs per songwriter for 50-60%+ accuracy

**Target Songwriters**: 6 top songwriters (focused dataset)

**APIs Used**:
- Genius API: Lyrics and songwriter credits
- Last.fm API: Popularity metrics and tags

**Expected Output**: songs_data_final.csv with 480-600+ songs
**Target per songwriter**: 80-100 songs minimum

## Step 1: Import Libraries and Setup

In [9]:
# Import essential libraries
import lyricsgenius as lg
import pylast
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully
Pandas version: 2.2.3
NumPy version: 2.2.2


## Step 2: Configure API Credentials

In [10]:
# Import API credentials
from config import GENIUS_TOKEN, LASTFM_API_KEY, LASTFM_API_SECRET

# Initialize Genius API
genius = lg.Genius(
    GENIUS_TOKEN,
    skip_non_songs=True,
    remove_section_headers=True,
    verbose=False,
    timeout=15,
    retries=3
)

# Initialize Last.fm API
lastfm_network = pylast.LastFMNetwork(
    api_key=LASTFM_API_KEY,
    api_secret=LASTFM_API_SECRET
)

print("API clients initialized successfully")
print(f"Genius Token: {GENIUS_TOKEN[:20]}...")
print(f"Last.fm API Key: {LASTFM_API_KEY[:20]}...")

API clients initialized successfully
Genius Token: z2UefZTuteHUDbYCjXil...
Last.fm API Key: 595d78b760f83e24e7cb...


## Step 3: Test API Connections

In [11]:
# Test Genius API
print("Testing Genius API connection...")
print("="*60)

try:
    test_song = genius.search_song("Blinding Lights", "The Weeknd")
    if test_song:
        print("[SUCCESS] Genius API connection working")
        print(f"Song: {test_song.title}")
        print(f"Artist: {test_song.artist}")
        print(f"Lyrics length: {len(test_song.lyrics)} characters")
        
        # Check for writer credits
        if hasattr(test_song, '_body') and 'writer_artists' in test_song._body:
            writers = [w['name'] for w in test_song._body['writer_artists']]
            print(f"Writers: {', '.join(writers)}")
    else:
        print("[ERROR] Song not found")
except Exception as e:
    print(f"[ERROR] {str(e)}")

print("\n" + "="*60)

# Test Last.fm API
print("Testing Last.fm API connection...")
print("="*60)

try:
    track = lastfm_network.get_track("The Weeknd", "Blinding Lights")
    print("[SUCCESS] Last.fm API connection working")
    print(f"Track: {track.get_name()}")
    print(f"Artist: {track.get_artist()}")
    print(f"Playcount: {track.get_playcount():,}")
    print(f"Listeners: {track.get_listener_count():,}")
except Exception as e:
    print(f"[ERROR] {str(e)}")

print("\n" + "="*60)
print("API connection tests complete")

Testing Genius API connection...
[SUCCESS] Genius API connection working
Song: Blinding Lights
Artist: The Weeknd
Lyrics length: 1266 characters
Writers: The Weeknd, Oscar Holter, Max Martin, Belly, DaHeala

Testing Last.fm API connection...
[SUCCESS] Last.fm API connection working
Track: Blinding Lights
Artist: The Weeknd
Playcount: 36,405,579
[SUCCESS] Genius API connection working
Song: Blinding Lights
Artist: The Weeknd
Lyrics length: 1266 characters
Writers: The Weeknd, Oscar Holter, Max Martin, Belly, DaHeala

Testing Last.fm API connection...
[SUCCESS] Last.fm API connection working
Track: Blinding Lights
Artist: The Weeknd
Playcount: 36,405,579
Listeners: 2,216,661

API connection tests complete
Listeners: 2,216,661

API connection tests complete


## Step 4: Define Target Songwriters (Expanded List)

In [12]:
# FOCUSED songwriter list - Top 6 songwriters with EXPANDED artist lists
# Target: 80-100 songs per songwriter for better ML accuracy
target_writers = {
    'Jack Antonoff': {
        'known_for': '80s-inspired production, indie-pop, emotional storytelling',
        'notable_artists': ['Taylor Swift', 'Lorde', 'Lana Del Rey', 'Bleachers', 
                           'St. Vincent', 'Carly Rae Jepsen', 'The Chicks', 'Clairo',
                           'Pink', 'Troye Sivan', 'Lana Del Rey', 'Sara Bareilles']
    },
    'Max Martin': {
        'known_for': 'Pop perfection, catchy hooks, radio-friendly hits',
        'notable_artists': ['Taylor Swift', 'The Weeknd', 'Ariana Grande', 'Katy Perry', 
                           'Maroon 5', 'P!nk', 'Backstreet Boys', 'Britney Spears',
                           'Demi Lovato', 'Bon Jovi', 'Usher', 'Ellie Goulding']
    },
    'Dr. Luke': {
        'known_for': 'Electronic pop, club bangers, provocative themes',
        'notable_artists': ['Katy Perry', 'Kesha', 'Doja Cat', 'Kim Petras', 
                           'P!nk', 'Miley Cyrus', 'Avril Lavigne', 'Taio Cruz',
                           'Flo Rida', 'B.o.B', 'Nicki Minaj', 'Pitbull']
    },
    'Ryan Tedder': {
        'known_for': 'Anthemic choruses, piano-driven, emotional crescendos',
        'notable_artists': ['Beyoncé', 'Adele', 'Taylor Swift', 'OneRepublic', 
                           'Leona Lewis', 'Ed Sheeran', 'Jonas Brothers', 'U2',
                           'Colbie Caillat', 'James Blunt', 'Ariana Grande', 'Camila Cabello']
    },
    'Sia Furler': {
        'known_for': 'Powerful vocals, emotional depth, EDM collaboration',
        'notable_artists': ['Rihanna', 'Beyoncé', 'David Guetta', 'Sia', 
                           'Flo Rida', 'Eminem', 'Christina Aguilera', 'Katy Perry',
                           'Britney Spears', 'Céline Dion', 'Rita Ora', 'Zayn']
    },
    'Stargate': {
        'known_for': 'R&B grooves, pop-soul fusion, smooth production',
        'notable_artists': ['Rihanna', 'Beyoncé', 'Ne-Yo', 'Katy Perry', 
                           'Sam Smith', 'Wiz Khalifa', 'Coldplay', 'Lionel Richie',
                           'Shakira', 'Mariah Carey', 'Jennifer Lopez', 'Usher']
    }
}

print("ENHANCED Data Collection Strategy")
print("="*60)
print("FOCUSED APPROACH: Top 6 songwriters with expanded artist lists")
print("Target: 80-100 songs per songwriter (vs previous 50)")
print("Why: ML models need 60-80+ samples per class for good accuracy")
print("\n" + "="*60)

for writer, info in target_writers.items():
    print(f"\n{writer}")
    print(f"  Style: {info['known_for']}")
    print(f"  Artists ({len(info['notable_artists'])}): {', '.join(info['notable_artists'][:5])}...")

print("\n" + "="*60)
print(f"Total Songwriters: {len(target_writers)}")
print(f"Target per songwriter: 80-100 songs")
print(f"Expected Total: 480-600 songs")
print(f"Expected Accuracy after training: 50-65%")

ENHANCED Data Collection Strategy
FOCUSED APPROACH: Top 6 songwriters with expanded artist lists
Target: 80-100 songs per songwriter (vs previous 50)
Why: ML models need 60-80+ samples per class for good accuracy


Jack Antonoff
  Style: 80s-inspired production, indie-pop, emotional storytelling
  Artists (12): Taylor Swift, Lorde, Lana Del Rey, Bleachers, St. Vincent...

Max Martin
  Style: Pop perfection, catchy hooks, radio-friendly hits
  Artists (12): Taylor Swift, The Weeknd, Ariana Grande, Katy Perry, Maroon 5...

Dr. Luke
  Style: Electronic pop, club bangers, provocative themes
  Artists (12): Katy Perry, Kesha, Doja Cat, Kim Petras, P!nk...

Ryan Tedder
  Style: Anthemic choruses, piano-driven, emotional crescendos
  Artists (12): Beyoncé, Adele, Taylor Swift, OneRepublic, Leona Lewis...

Sia Furler
  Style: Powerful vocals, emotional depth, EDM collaboration
  Artists (12): Rihanna, Beyoncé, David Guetta, Sia, Flo Rida...

Stargate
  Style: R&B grooves, pop-soul fusion, smoo

## Step 5: Helper Functions for Data Collection

In [13]:
def extract_song_data(song, target_songwriter):
    """
    Extract comprehensive data from a Genius song object
    
    Args:
        song: Genius song object
        target_songwriter: Name of the songwriter we're collecting for
    
    Returns:
        dict: Song data or None if invalid
    """
    if not song or not hasattr(song, 'lyrics'):
        return None
    
    # Skip songs without lyrics or very short lyrics
    if not song.lyrics or len(song.lyrics.strip()) < 100:
        return None
    
    song_data = {
        'title': song.title,
        'artist': song.artist,
        'lyrics': song.lyrics,
        'target_songwriter': target_songwriter,
        'writers': 'Unknown',
        'producers': 'Unknown',
        'release_date': None,
        'url': song.url if hasattr(song, 'url') else None,
        'pageviews': 0,
        'lastfm_playcount': 0,
        'lastfm_listeners': 0,
        'lastfm_tags': ''
    }
    
    # Extract metadata from Genius
    if hasattr(song, '_body'):
        metadata = song._body
        
        # Get writers
        if 'writer_artists' in metadata:
            writers = [w['name'] for w in metadata['writer_artists']]
            song_data['writers'] = ', '.join(writers)
        
        # Get producers
        if 'producer_artists' in metadata:
            producers = [p['name'] for p in metadata['producer_artists']]
            song_data['producers'] = ', '.join(producers)
        
        # Get release date
        if 'release_date' in metadata and metadata['release_date']:
            song_data['release_date'] = metadata['release_date']
        
        # Get pageviews
        if 'stats' in metadata and 'pageviews' in metadata['stats']:
            song_data['pageviews'] = metadata['stats']['pageviews']
    
    return song_data


def get_lastfm_data(artist_name, song_title):
    """
    Get popularity metrics from Last.fm
    
    Args:
        artist_name: Artist name
        song_title: Song title
    
    Returns:
        dict: Last.fm metrics
    """
    lastfm_data = {
        'playcount': 0,
        'listeners': 0,
        'tags': ''
    }
    
    try:
        track = lastfm_network.get_track(artist_name, song_title)
        lastfm_data['playcount'] = track.get_playcount()
        lastfm_data['listeners'] = track.get_listener_count()
        
        # Get tags
        tags = track.get_top_tags(limit=3)
        lastfm_data['tags'] = ', '.join([tag.item.get_name() for tag in tags])
    except:
        pass
    
    return lastfm_data


def save_checkpoint(songwriter_name, songs_data):
    """
    Save checkpoint for a songwriter's collected data
    
    Args:
        songwriter_name: Name of songwriter
        songs_data: List of song dictionaries
    """
    if not os.path.exists('data'):
        os.makedirs('data')
    
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    df = pd.DataFrame(songs_data)
    df.to_csv(checkpoint_file, index=False)
    print(f"  [CHECKPOINT] Saved {len(songs_data)} songs to {checkpoint_file}")


print("Helper functions defined successfully")
print("  - extract_song_data(): Extract metadata from Genius")
print("  - get_lastfm_data(): Get popularity metrics")
print("  - save_checkpoint(): Save progress during collection")

Helper functions defined successfully
  - extract_song_data(): Extract metadata from Genius
  - get_lastfm_data(): Get popularity metrics
  - save_checkpoint(): Save progress during collection


## Step 6: HIGH-VOLUME Data Collection (80-100 Songs Per Songwriter)

In [14]:
# HIGH-VOLUME data collection - targeting 80-100 songs per songwriter
print("STARTING HIGH-VOLUME DATA COLLECTION")
print("="*60)
print(f"Target: {len(target_writers)} songwriters (FOCUSED APPROACH)")
print(f"Goal: 80-100 songs per songwriter (INCREASED from 50)")
print(f"Expected total: 480-600 songs")
print(f"Why: ML models need 60-80+ samples per class for good accuracy")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

# Storage
all_songs_data = []
songwriter_stats = {}

# Collection parameters - ENHANCED FOR HIGHER VOLUME
SONGS_PER_SONGWRITER_TARGET = 100  # Increased from 50
MAX_SONGS_PER_ARTIST = 20          # Increased from 15
RATE_LIMIT_DELAY = 1.5             # Seconds between songs
ARTIST_DELAY = 3.0                 # Seconds between artists
RATE_LIMIT_WAIT = 60               # Seconds to wait if rate limited

# Load existing checkpoints to resume if interrupted
for songwriter_name in target_writers.keys():
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    if os.path.exists(checkpoint_file):
        existing_df = pd.read_csv(checkpoint_file)
        existing_songs = existing_df.to_dict('records')
        all_songs_data.extend(existing_songs)
        songwriter_stats[songwriter_name] = len(existing_songs)
        print(f"[RESUME] Loaded {len(existing_songs)} existing songs for {songwriter_name}")

# Collect songs for each songwriter
for songwriter_name, writer_info in target_writers.items():
    print(f"\n{'='*60}")
    print(f"COLLECTING FOR: {songwriter_name}")
    print(f"{'='*60}")
    print(f"Style: {writer_info['known_for']}")
    print(f"Target artists: {len(writer_info['notable_artists'])}")
    
    # Get existing songs count
    existing_count = songwriter_stats.get(songwriter_name, 0)
    print(f"Existing songs: {existing_count}")
    print(f"Target: {SONGS_PER_SONGWRITER_TARGET} songs")
    print(f"Need to collect: {max(0, SONGS_PER_SONGWRITER_TARGET - existing_count)} more songs")
    
    if existing_count >= SONGS_PER_SONGWRITER_TARGET:
        print(f"[SKIP] Already have enough songs for {songwriter_name}")
        continue
    
    songwriter_songs = []
    
    # Load existing songs if checkpoint exists
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    if os.path.exists(checkpoint_file):
        existing_df = pd.read_csv(checkpoint_file)
        songwriter_songs = existing_df.to_dict('records')
    
    songs_collected = len(songwriter_songs)
    
    # Search through each artist
    for artist_idx, artist_name in enumerate(writer_info['notable_artists'], 1):
        if songs_collected >= SONGS_PER_SONGWRITER_TARGET:
            print(f"\n[TARGET REACHED] Collected {songs_collected} songs for {songwriter_name}")
            break
        
        print(f"\n[{artist_idx}/{len(writer_info['notable_artists'])}] Searching: {artist_name}")
        
        try:
            # Search for artist on Genius
            artist = genius.search_artist(artist_name, max_songs=MAX_SONGS_PER_ARTIST)
            
            if not artist:
                print(f"  [NOT FOUND] Artist '{artist_name}' not found")
                time.sleep(ARTIST_DELAY)
                continue
            
            print(f"  Found artist: {artist.name} ({len(artist.songs)} songs)")
            
            # Process each song
            for song_idx, song in enumerate(artist.songs, 1):
                if songs_collected >= SONGS_PER_SONGWRITER_TARGET:
                    break
                
                try:
                    # Extract song data
                    song_data = extract_song_data(song, songwriter_name)
                    
                    if not song_data:
                        continue
                    
                    # Check if songwriter is credited
                    writers_list = song_data['writers'].lower()
                    songwriter_name_check = songwriter_name.lower()
                    
                    # Special handling for names
                    name_variants = [songwriter_name_check]
                    if ' ' in songwriter_name_check:
                        # Try last name only
                        last_name = songwriter_name_check.split()[-1]
                        name_variants.append(last_name)
                    
                    is_credited = any(variant in writers_list for variant in name_variants)
                    
                    if not is_credited:
                        continue
                    
                    # Check for duplicates
                    is_duplicate = any(
                        s['title'].lower() == song_data['title'].lower() and 
                        s['artist'].lower() == song_data['artist'].lower()
                        for s in songwriter_songs
                    )
                    
                    if is_duplicate:
                        continue
                    
                    # Get Last.fm data
                    lastfm_data = get_lastfm_data(song_data['artist'], song_data['title'])
                    song_data['lastfm_playcount'] = lastfm_data['playcount']
                    song_data['lastfm_listeners'] = lastfm_data['listeners']
                    song_data['lastfm_tags'] = lastfm_data['tags']
                    
                    # Add song
                    songwriter_songs.append(song_data)
                    songs_collected += 1
                    
                    print(f"    [{songs_collected}/{SONGS_PER_SONGWRITER_TARGET}] Added: {song_data['title']} by {song_data['artist']}")
                    
                    # Rate limiting
                    time.sleep(RATE_LIMIT_DELAY)
                    
                except Exception as e:
                    print(f"    [ERROR] Processing song: {str(e)[:50]}")
                    continue
            
            # Delay between artists
            time.sleep(ARTIST_DELAY)
            
        except Exception as e:
            error_msg = str(e)
            if '429' in error_msg or 'rate limit' in error_msg.lower():
                print(f"  [RATE LIMIT] Waiting {RATE_LIMIT_WAIT} seconds...")
                time.sleep(RATE_LIMIT_WAIT)
            else:
                print(f"  [ERROR] Searching artist: {error_msg[:100]}")
            continue
    
    # Save checkpoint after each songwriter
    if songwriter_songs:
        save_checkpoint(songwriter_name, songwriter_songs)
        songwriter_stats[songwriter_name] = len(songwriter_songs)
    
    print(f"\n[COMPLETE] {songwriter_name}: {len(songwriter_songs)} songs collected")

# Combine all songs
print(f"\n{'='*60}")
print("COLLECTION SUMMARY")
print(f"{'='*60}")

all_songs_data = []
for songwriter_name in target_writers.keys():
    checkpoint_file = f"data/checkpoint_{songwriter_name.replace(' ', '_')}.csv"
    if os.path.exists(checkpoint_file):
        df = pd.read_csv(checkpoint_file)
        all_songs_data.extend(df.to_dict('records'))
        print(f"  {songwriter_name}: {len(df)} songs")

print(f"\nTotal songs collected: {len(all_songs_data)}")
print(f"Average per songwriter: {len(all_songs_data) / len(target_writers):.1f}")
print(f"Collection completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}")

STARTING HIGH-VOLUME DATA COLLECTION
Target: 6 songwriters (FOCUSED APPROACH)
Goal: 80-100 songs per songwriter (INCREASED from 50)
Expected total: 480-600 songs
Why: ML models need 60-80+ samples per class for good accuracy
Started: 2025-11-05 14:29:16
[RESUME] Loaded 50 existing songs for Jack Antonoff
[RESUME] Loaded 46 existing songs for Max Martin
[RESUME] Loaded 41 existing songs for Dr. Luke
[RESUME] Loaded 24 existing songs for Ryan Tedder
[RESUME] Loaded 20 existing songs for Sia Furler
[RESUME] Loaded 22 existing songs for Stargate

COLLECTING FOR: Jack Antonoff
Style: 80s-inspired production, indie-pop, emotional storytelling
Target artists: 12
Existing songs: 50
Target: 100 songs
Need to collect: 50 more songs

[1/12] Searching: Taylor Swift
  Found artist: Taylor Swift (20 songs)
  Found artist: Taylor Swift (20 songs)
    [51/100] Added: Cruel Summer by Taylor Swift
    [51/100] Added: Cruel Summer by Taylor Swift
    [52/100] Added: august by Taylor Swift
    [52/100] Ad

## Step 7: Save Final Dataset

In [15]:
# Create DataFrame from collected data
df_songs = pd.DataFrame(all_songs_data)

print("Dataset Overview")
print("="*60)
print(f"Total Songs: {len(df_songs)}")
print(f"Total Columns: {len(df_songs.columns)}")
print(f"\nColumns: {list(df_songs.columns)}")

print("\nSongs per Songwriter:")
print(df_songs['target_songwriter'].value_counts())

print("\nDataset Statistics:")
print(f"  Average lyrics length: {df_songs['lyrics'].str.len().mean():.0f} characters")
print(f"  Median lyrics length: {df_songs['lyrics'].str.len().median():.0f} characters")
print(f"  Missing values: {df_songs.isnull().sum().sum()}")

# Save to CSV
output_file = 'data/songs_data_final.csv'
df_songs.to_csv(output_file, index=False)
print(f"\n[SAVED] Dataset saved to {output_file}")

# Also save as JSON backup
json_file = 'data/songs_data_final.json'
df_songs.to_json(json_file, orient='records', indent=2)
print(f"[SAVED] JSON backup saved to {json_file}")

# Save metadata
metadata = {
    'collection_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_songs': len(df_songs),
    'songwriters': list(songwriter_stats.keys()),
    'songs_per_songwriter': songwriter_stats
}

import json
metadata_file = 'data/collection_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"[SAVED] Metadata saved to {metadata_file}")

print("\n" + "="*60)
print("DATA COLLECTION NOTEBOOK COMPLETE")
print("="*60)
print(f"\nNext Step: Run 02_preprocessing.ipynb")

Dataset Overview
Total Songs: 282
Total Columns: 12

Columns: ['title', 'artist', 'lyrics', 'target_songwriter', 'writers', 'producers', 'release_date', 'url', 'pageviews', 'lastfm_playcount', 'lastfm_listeners', 'lastfm_tags']

Songs per Songwriter:
target_songwriter
Jack Antonoff    75
Dr. Luke         67
Max Martin       62
Ryan Tedder      34
Stargate         24
Sia Furler       20
Name: count, dtype: int64

Dataset Statistics:
  Average lyrics length: 1949 characters
  Median lyrics length: 1870 characters
  Missing values: 173

[SAVED] Dataset saved to data/songs_data_final.csv
[SAVED] JSON backup saved to data/songs_data_final.json
[SAVED] Metadata saved to data/collection_metadata.json

DATA COLLECTION NOTEBOOK COMPLETE

Next Step: Run 02_preprocessing.ipynb


In [16]:
# Display sample of collected data
print("Sample of Collected Data:")
print("="*60)
df_songs[['title', 'artist', 'target_songwriter', 'writers', 'lastfm_playcount']].head(10)

Sample of Collected Data:


,title,artist,target_songwriter,writers,lastfm_playcount
0,All Too Well (10 Minute Version) (Taylor’s Ver...,Taylor Swift,Jack Antonoff,"Liz Rose, Taylor Swift",104
1,Fortnight,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff, Post Malone",200274
2,The Tortured Poets Department,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",12304018
3,Lover,Taylor Swift,Jack Antonoff,Taylor Swift,26306242
4,But Daddy I Love Him,Taylor Swift,Jack Antonoff,"Taylor Swift, Aaron Dessner",12896652
5,Down Bad,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",16556853
6,Is It Over Now? (Taylor’s Version) [From the V...,Taylor Swift,Jack Antonoff,"Taylor Swift, Jack Antonoff",39
7,Liability,Lorde,Jack Antonoff,"Jack Antonoff, Lorde",17910966
8,Green Light,Lorde,Jack Antonoff,"Joel Little, Jack Antonoff, Lorde",19125306
9,The Louvre,Lorde,Jack Antonoff,"Jack Antonoff, Lorde",11939649
